<a href="https://colab.research.google.com/github/rezzz59/Sentimen-Analysis-Aplikasi-Grab/blob/main/grab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install google-play-scrapper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.1 MB/s eta 0:00:00


In [11]:
import sys
!{sys.executable} -m pip install google-play-scraper



In [55]:
import pandas as pd
from google_play_scraper import Sort, reviews

result, countinuation_token = reviews(
    'com.grabtaxi.passenger',
    lang = 'id',
    country = 'id',
    sort = Sort.NEWEST,
    count= 51000,
)

df = pd.DataFrame(result)

df = df[['content', 'score']]

df.to_csv('grab_review_raw.csv', index = False)

In [57]:
%pip install sastrawi

In [58]:
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

slang_dict = {
    "ga": "tidak", "gak": "tidak", "gakk": "tidak", "nggak": "tidak",
    "yg": "yang", "dr": "dari", "bgt": "banget", "kl": "kalau",
    "udh": "sudah", "udah": "sudah", "aja": "saja", "jd": "jadi",
    "tp": "tapi", "pake": "pakai", "sdh": "sudah", "aja": "saja",
    "dapet": "dapat", "ilang": "hilang", "lemot": "lambat", "gercep": "cepat",
}

def clean_text(text):
  text = text.lower()

  #menghapus link, mention dan tag
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
  text = text = re.sub(r'@\w+|\#', '', text)

  #menghapus angka dan tanda bca
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'\d+', '', text)

  #menghapus emoji
  text = text.encode('ascii', 'ignore').decode('ascii')

  #normalisasi kata & tokenizing
  words = text.split()
  cleaned_words = [slang_dict.get(w, w) for w in words]

  return " ".join(cleaned_words)

df['content_cleaned'] = df['content'].apply(clean_text)
df = df[df['content_cleaned'] != '']


In [96]:
def labeling(score):
  if score > 3:
    return "positif"
  if score <= 2:
    return 'negatif'
  else:
    return 'netral'

df['label'] = df['score'].apply(labeling)

/tmp/ipykernel_2795/272204828.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['score'].apply(labeling)


In [97]:
df['label'].value_counts()

,count
label,
positif,38071
negatif,10525
netral,1612


In [98]:
from sklearn.utils import resample
target_samples = 8977

df_pos_bal = df[df['label'] == 'positif'].sample(target_samples, random_state=42)
df_neg_bal = df[df['label'] == 'negatif']
df_neu_bal = resample(df[df['label'] == 'netral'], replace=True, n_samples=target_samples, random_state=42)

df_final = pd.concat([df_pos_bal, df_neg_bal, df_neu_bal])

In [101]:
df_final

,content,score,content_cleaned,label
29074,sangat puas,5,sangat puas,positif
44781,keren,5,keren,positif
2998,baik,4,baik,positif
40655,supirnya baik,5,supirnya baik,positif
28226,ok,5,ok,positif
...,...,...,...,...
28555,"bagus, penjemputan nya cepat",3,bagus penjemputan nya cepat,netral
41106,Udah bagus sih... Cuman masih aja ada lag pas ...,3,sudah bagus sih cuman masih saja ada lag pas a...,netral
23322,i like,3,i like,netral
20662,susah dapet driver klo jam malam padahal gw be...,3,susah dapat driver klo jam malam padahal gw be...,netral


In [102]:
df_final['label'].value_counts()

,count
label,
negatif,10525
positif,8977
netral,8977


In [103]:
df_final[['content_cleaned', 'score', 'label']]

,content_cleaned,score,label
29074,sangat puas,5,positif
44781,keren,5,positif
2998,baik,4,positif
40655,supirnya baik,5,positif
28226,ok,5,positif
...,...,...,...
28555,bagus penjemputan nya cepat,3,netral
41106,sudah bagus sih cuman masih saja ada lag pas a...,3,netral
23322,i like,3,netral
20662,susah dapat driver klo jam malam padahal gw be...,3,netral


In [124]:
from sklearn.model_selection import train_test_split

X = df_final['content_cleaned'].astype(str).values
y = pd.get_dummies(df_final['label']).astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)


In [125]:
y

array([[0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 1, 0],
       [0, 1, 0],
       [0, 1, 0]])

In [126]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

#menyamakan panjang kalimat ada yang panjang dan pendek, sehingga disamakan menjadi 100 kata
X_train_pad = pad_sequences(X_train_seq, maxlen=100, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=100, padding='post', truncating='post')

In [127]:
df_final

,content,score,content_cleaned,label
29074,sangat puas,5,sangat puas,positif
44781,keren,5,keren,positif
2998,baik,4,baik,positif
40655,supirnya baik,5,supirnya baik,positif
28226,ok,5,ok,positif
...,...,...,...,...
28555,"bagus, penjemputan nya cepat",3,bagus penjemputan nya cepat,netral
41106,Udah bagus sih... Cuman masih aja ada lag pas ...,3,sudah bagus sih cuman masih saja ada lag pas a...,netral
23322,i like,3,i like,netral
20662,susah dapet driver klo jam malam padahal gw be...,3,susah dapat driver klo jam malam padahal gw be...,netral


In [128]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional

model = Sequential([
    Embedding(input_dim=10000, output_dim=100),

    SpatialDropout1D(0.3),
    Bidirectional(LSTM(64, return_sequences=True)),
    LSTM(100, dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_14 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_13            │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [129]:


# Callback untuk berhenti otomatis jika akurasi sudah memuaskan
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('accuracy') > 0.93 and logs.get('val_accuracy') > 0.93):
            print("\nAkurasi sudah mencapai 93%")
            self.model.stop_training = True

callbacks = myCallback()

# Mulai Pelatihan
history = model.fit(
    X_train_pad, y_train,
    epochs=40,               # Maksimal putaran belajar
    batch_size=64,           # Jumlah data yang diproses sekali jalan
    validation_data=(X_test_pad, y_test), # Ujian langsung setiap selesai 1 epoch
    callbacks=[callbacks],
    verbose=1
)

Epoch 1/40
356/356 ━━━━━━━━━━━━━━━━━━━━ 96s 258ms/step - accuracy: 0.3667 - loss: 1.0972 - val_accuracy: 0.3720 - val_loss: 1.0954
Epoch 2/40
356/356 ━━━━━━━━━━━━━━━━━━━━ 96s 269ms/step - accuracy: 0.3664 - loss: 1.0967 - val_accuracy: 0.3720 - val_loss: 1.0953
Epoch 3/40
356/356 ━━━━━━━━━━━━━━━━━━━━ 94s 265ms/step - accuracy: 0.3689 - loss: 1.0962 - val_accuracy: 0.3720 - val_loss: 1.0956
Epoch 4/40
356/356 ━━━━━━━━━━━━━━━━━━━━ 93s 260ms/step - accuracy: 0.3686 - loss: 1.0962 - val_accuracy: 0.3720 - val_loss: 1.0954
Epoch 5/40
356/356 ━━━━━━━━━━━━━━━━━━━━ 93s 260ms/step - accuracy: 0.3690 - loss: 1.0960 - val_accuracy: 0.3720 - val_loss: 1.0953
Epoch 6/40
356/356 ━━━━━━━━━━━━━━━━━━━━ 147s 273ms/step - accuracy: 0.3690 - loss: 1.0959 - val_accuracy: 0.3720 - val_loss: 1.0955
Epoch 7/40
144/356 ━━━━━━━━━━━━━━━━━━━━ 56s 269ms/step - accuracy: 0.3610 - loss: 1.0971

KeyboardInterrupt: 

In [123]:
# Cek 1 baris data yang masuk ke model
print("Teks Asli:", df_final['content_cleaned'].iloc[0])
print("Hasil Sequence:", X_train.iloc[0])

Teks Asli: sangat puas


KeyError: 0